
read in the iza database stored as a feather file and the corresponding pickle of ase atoms objects

In [15]:
import pandas as pd 
import pickle 
import os 
os.chdir('C:/Users/Arun Gopalan/OneDrive - The University of Manchester/my_cloned_repos/mofography/examples')
# Load the model from the file
iza_atoms = pickle.load(open('iza_atoms_list.pkl', 'rb'))
df = pd.read_feather('iza_df.feather')

start a dask cluster locally

In [16]:
from dask import delayed 
from dask.distributed import LocalCluster, Client, progress

cluster = LocalCluster()
client = Client(cluster)
client


c:\Users\Public\anaconda3\envs\jl_env\lib\site-packages\distributed\node.py:183: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 60795 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:60795/status,
Dashboard: http://127.0.0.1:60795/status,Workers: 4
Total threads: 16,Total memory: 15.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:60796,Workers: 4
Dashboard: http://127.0.0.1:60795/status,Total threads: 16
Started: Just now,Total memory: 15.73 GiB
Comm: tcp://127.0.0.1:60836,Total threads: 4
Dashboard: http://127.0.0.1:60837/status,Memory: 3.93 GiB
Nanny: tcp://127.0.0.1:60799,


Import mofography after adding its path

In [17]:
import sys 

path_to_mgr = os.path.abspath('C:/Users/Arun Gopalan/OneDrive - The University of Manchester/my_cloned_repos/mofography')
sys.path.append(path_to_mgr)

import mofography as mgr

create a workflow function

In [18]:

#* create a workflow 

import numpy as np
import ase
import dask.array as da
from ase.data import covalent_radii
from scipy.ndimage import zoom
from skimage.measure import regionprops
import random
spacing = 0.5
wall_thickness =1.0
mask_thickness =1.5
bins = np.linspace(0,30, 50)
block_size = 20000
a, b, c, =1,1,1 
distance_threshold = 0.25

def get_rag_from_atoms(mat_atoms, spacing=spacing, wall_thickness=wall_thickness, mask_thickness=mask_thickness, distance_threshold = distance_threshold, bins=bins, block_size=block_size, a=a, b=b, c=c):
    try:
        radii = covalent_radii[mat_atoms.get_atomic_numbers()]
        grid = mgr.dgrid_from_atoms(mat_atoms, radii=radii, spacing =spacing, block_size=block_size)
        # bins = np.linspace(0,grid.compute().max(), 50)
        mat_atoms = ase.build.make_supercell(mat_atoms, [[a,0,0],[0,b,0],[0,0,
        c]])
        grid = da.from_array(np.tile(grid.compute(), [a,b,c]))
        grid = da.from_array(zoom(grid.compute(), 2,mode='grid-wrap', order=1 ), chunks=(40,40,40))
        regions, maxima = mgr.regions_from_dgrid(grid.compute(), mask_thickness=mask_thickness, compactness=0.001, h=0.2)
        maxima_radii = grid.compute()[tuple(maxima.T)]
        # pbc_groups = mgr.apply_pbc(regions, maxima, maxima_radii, wall_thickness=1, return_conn=True)
        pbc_groups = mgr.apply_pbc(regions, maxima, maxima_radii,mat_atoms, wall_thickness=wall_thickness, return_conn=True, minimum_overlap_fraction=0.5)
        rag2 = mgr.get_rag_from_regions_and_grid(regions, grid.compute(),mat_atoms,maxima)
        # rag2.graph['name'] = mat_atoms.info['name']
        # pbc_groups = mgr.get_periodic_map(regions=regions, maxima=maxima, maxima_radii=grid.compute()[tuple(maxima.T)], return_conn=True, wall_thickness=4)
        props = regionprops(regions)
        rag2 = mgr.add_psd_to_rag( rag2,region_props=props,  bins = bins, ase_atoms = mat_atoms, shape =regions.shape)
        rag2 = mgr.add_psd_to_rag_with_pbc(rag = rag2, props = props, pbc_groups= pbc_groups, bins = bins, ase_atoms = mat_atoms, shape =regions.shape)
        # connections = mgr.get_connections_for_rag(rag2, regions, maxima, mat_atoms, grid.compute())
        # rag2 = mgr.add_clustering_to_rag(rag2, affinity='euclidean', linkage='single', distance_threshold=0.25, no_pbc=False)
        rag2 = mgr.add_clustering_to_rag_pca(rag2, kwargs_cluster=dict(distance_threshold=distance_threshold, kernel = 'linear'))#, affinity='euclidean', linkage='single', distance_threshold=0.25, no_pbc=False)

        return rag2
    except:
        return 'NaN'

map workflow to a list of ase atoms objects for the IZA dataset

In [19]:
#* map things using dask delayed 

from dask import delayed 
graphs = []

for i in range(len(iza_atoms)):
    graphs.append(delayed(get_rag_from_atoms)(iza_atoms[i]))

run the workflow and track it's progress

In [20]:
# * run the workflwo
import mofography as mgr
futures = client.persist(graphs)
progress(futures)

VBox()

Check how any of the calculations failed by looking of 'NaNs'

In [21]:

# * get the results and check for NaNs
results = [f.compute() for f in futures]

import numpy as np
np.where([r=='NaN' for r in results])

Test the clustering by visualizing a random structure 

In [ ]:
import random 
import numpy as np
chosen = random.choice(np.arange(len(iza_atoms)))
# chosen = np.random.choice(len(iza_atoms))
print(chosen, df['name'][chosen])

75 IFR-0


In [14]:

import numpy as np
import ase
import dask.array as da
from ase.data import covalent_radii
from scipy.ndimage import zoom
from skimage.measure import regionprops
import random
spacing = 0.5
mask_thickness =1.5
wall_thickness =1.0
bins = np.linspace(0,30, 50)
block_size = 20000
a, b, c, =1,1,1 
distance_threshold = 0.25
import random 
# chose one at random 


# or set one manually for testing
# chosen = 195
mat_atoms = iza_atoms[chosen]
mat_updated = mgr.update_with_covalent_bonds(mat_atoms)
radii = covalent_radii[mat_atoms.get_atomic_numbers()]
grid = mgr.dgrid_from_atoms(mat_atoms, radii=radii, spacing =spacing, block_size=block_size)
# bins = np.linspace(0,grid.compute().max(), 50)
mat_atoms = ase.build.make_supercell(mat_atoms, [[a,0,0],[0,b,0],[0,0,
c]])
grid = da.from_array(np.tile(grid.compute(), [a,b,c]))
grid = da.from_array(zoom(grid.compute(), 2,mode='grid-wrap', order=1 ), chunks=(40,40,40))
regions, maxima = mgr.regions_from_dgrid(grid.compute(), mask_thickness=mask_thickness, compactness=0.001, h=0.2)
maxima_radii = grid.compute()[tuple(maxima.T)]
# pbc_groups = mgr.apply_pbc(regions, maxima, maxima_radii, wall_thickness=1, return_conn=True)
pbc_groups = mgr.apply_pbc(regions, maxima, maxima_radii,mat_atoms, wall_thickness=wall_thickness, return_conn=True, minimum_overlap_fraction=0.5)
rag2 = mgr.get_rag_from_regions_and_grid(regions, grid.compute(),mat_atoms,maxima)
rag2 = mgr.add_pbc_to_rag(rag2, pbc_groups)
# rag2.graph['name'] = mat_atoms.info['name']
# pbc_groups = mgr.get_periodic_map(regions=regions, maxima=maxima, maxima_radii=grid.compute()[tuple(maxima.T)], return_conn=True, wall_thickness=4)
props = regionprops(regions)
rag2 = mgr.add_psd_to_rag( rag2,region_props=props,  bins = bins, ase_atoms = mat_atoms, shape =regions.shape)
rag2 = mgr.add_psd_to_rag_with_pbc(rag = rag2, props = props, pbc_groups= pbc_groups, bins = bins, ase_atoms = mat_atoms, shape =regions.shape)
# connections = mgr.get_connections_for_rag(rag2, regions, maxima, mat_atoms, grid.compute())
# rag2 = mgr.add_clustering_to_rag(rag2, affinity='euclidean', linkage='single', distance_threshold=0.25, no_pbc=False)
rag2 = mgr.add_clustering_to_rag_pca(rag2, kwargs_cluster=dict(distance_threshold=distance_threshold, kernel = 'linear'))#, af


#* this part is for visualizing the types of pockets
regions_cluster = np.zeros(regions.shape)
for i in range(1, len(rag2.nodes)+1):
    regions_cluster[np.where(regions==rag2.nodes[i]['labels'][0])] = int(rag2.nodes[i]['cluster'])+1

# add energy histograms to the nodes 
forcefield_mixed = mgr.mix_lorentz_berthelot(mat_atoms)
rag2 = mgr.add_vdw_hist_to_rag_with_pbc(rag2, props, pbc_groups, mat_atoms, forcefield_mixed, regions.shape)




KeyboardInterrupt: 

Make a figure with three key traces 
- pore surface colored by raw segmentation
- pore surface colored by pbc grouped segments
- pore surface colored by the types of segments as detected by clustering

In [13]:

import plotly.graph_objects as go
import plotly.io as pio 
pio.renderers.default = "browser"
import plotly.express as px

cscale  = px.colors.qualitative.Pastel
regions_pbc= mgr.apply_pbc(regions, maxima, maxima_radii,mat_atoms, wall_thickness=wall_thickness, return_conn=False, minimum_overlap_fraction=0.5)
mat_updated = mgr.update_with_covalent_bonds(mat_atoms)
frame_trace = mgr.plot_framework_as_mesh(mat_updated, show_rings=False, stylize=True)
# rag_trace=mgr.plot_rag(rag2,plot_with='maxima', name='p-graph') 
surface_trace=mgr.plot_surface_dgrid(grid.compute(), 1.75, select='all',ase_atoms=mat_updated,region_labels=regions, cscale = cscale , stylize=True, name = 'Raw segmentation', hover_label='Region :')
surface_trace_pbc=mgr.plot_surface_dgrid(grid.compute(), 1.75, select='all',ase_atoms=mat_updated,region_labels=regions_pbc, cscale = cscale , stylize=True, name='PBC groups', hover_label='PBC Group :')
surface_trace_cluster=mgr.plot_surface_dgrid(grid.compute(), 1.75, select='all',ase_atoms=mat_updated,region_labels=regions_cluster, cscale = cscale , stylize=True, name ='Cavity types', hover_label='Cavity type :')
# surface_trace=mgr.plot_surface_dgrid(grid.compute(), 1.75, select='all',ase_atoms=mat_updated,region_labels=regions_pbc, stylize=True)
# surface_trace=mgr.plot_surface_dgrid(grid.compute(), 1.75, select=list(pbc_groups[0]),ase_atoms=mat_updated,region_labels=regions, stylize=True)
cell_trace=mgr.plot_cell(mat_updated)
maxima_trace = mgr.plot_maxima(maxima = maxima, dgrid = grid.compute(),ase_atoms =mat_updated)



# fig = go.Figure(data=[*frame_trace,rag_trace,surface_trace,cell_trace,maxima_trace]).update(layout=mgr.white_layout).show()
# fig = go.Figure(data=[*frame_trace,surface_trace,surface_trace_pbc, surface_trace_cluster, cell_trace,maxima_trace]).update(layout=mgr.white_layout)
fig = go.Figure(data=[surface_trace, surface_trace_pbc, surface_trace_cluster, cell_trace,maxima_trace]).update(layout=mgr.white_layout)
# fig = go.Figure(data=[*frame_trace,surface_trace,surface_trace_pbc, surface_trace_cluster, cell_trace]).update(layout=mgr.white_layout)
fig.show(renderer="browser")

C:\Users\Public\anaconda3\envs\jl_env\lib\site-packages\ase\atoms.py:967: VisibleDeprecationWarning:

Use get_global_number_of_atoms() instead



save the figure as a html file with the same name as the material

In [14]:
fig.write_html(file=df['name'][chosen]+'.html')

In [15]:
fig.show()

In [18]:
import plotly.graph_objects as go
go.Figure(data= [*mgr.plot_psds(rag2, pbc_groups, cscale=cscale)]).update(layout= mgr.white_layout).update_layout(barmode='stack').show()
# go.Figure(data= [*mgr.plot_vdw_hists(rag2, pbc_groups, cscale=cscale)]).show()

In [7]:
def plot_psds(rag, pbc_groups, cscale=None, no_pbc=False):
    """
    > This function takes in a RAG and a list of PBC groups and plots the PSD for each PBC group. 
    
    Let's see how it works:
    
    :param rag: the region adjacency graph
    :param pbc_groups: a list of sets of nodes that are connected by periodic boundary conditions
    :return: A plotly figure object
    """

    import plotly.graph_objects as go
    import plotly.express as px 
    import networkx as nx 
    import numpy as np
    if cscale is None:
        cscale = px.colors.qualitative.Alphabet
    unique_clusters = np.unique(list(nx.get_node_attributes(rag,'cluster').values()))

    cmap = dict(zip(unique_clusters, cscale[:len(unique_clusters)]))
    traces = []
    if no_pbc:
        for node in list(rag.nodes().keys()): 
            # node = np.min(list(g))
            traces.append(go.Scatter(x=rag.nodes[node]['psd_x'], y=rag.nodes[node]['psd_y'], mode='markers+lines', line_color = cmap.get(rag.nodes[node]['cluster']), hovertext = 'Cluster: '+ rag.nodes[node]['cluster']+'</br>'+'Region: {}'.format(node) , name='Region: ' +str(rag.nodes[node]['labels'][0])))
    else:
        for i,g in enumerate(pbc_groups): 
            node = np.min(list(g))
            traces.append(go.Scatter(x=rag.nodes[node]['psd_x_pbc'], y=rag.nodes[node]['psd_y_pbc'], mode='markers+lines', line_color = cmap.get(rag.nodes[node]['cluster']), hovertext = 'Cluster: '+ rag.nodes[node]['cluster']+'</br>'+'Regions: {}'.format(','.join(map(str, list(g)))) , name= 'PBC group: {}'.format(i+1) ))
            traces.append(go.Bar(x=rag.nodes[node]['psd_x_pbc'], y=rag.nodes[node]['psd_y_pbc'], marker_color=cmap.get(rag.nodes[node]['cluster']), name='PBC group: {}'.format(i+1) ) )
    
    return traces

finally, shutdown the dask cluster

In [16]:
','.join(map(str, list(pbc_groups[0])))

'1,2'

In [ ]:
client.shutdown()

So we have to create a function that compute the energy histogram for each node in the pore graph

Compute histogram for a given region

In [19]:
egrid = mgr.egrid_from_atoms2(mat_atoms)

In [20]:
egrid.compute().min()

-1192.2544126166656

In [21]:
pbc_groups[0]

{1, 2, 3, 4, 6, 7, 8, 9}

In [8]:
def get_energies_for_group(group, prop, ase_atoms, forcefield_mixed, shape, cutoff=12.5):
    


    # * repeat the ase atoms and forcefield to meet cutoff    
    # nrepeats = np.prod(mgr.get_repetitions_to_meet_cutoff(ase_atoms, cutoff))
    ase_atoms_repeat, forcefield_mixed_repeat = mgr.repeat_to_meet_cutoff(ase_atoms, forcefield_mixed, cutoff)
    # import numpy as np  

    # * get aabb object on repeated system
    ab_repeat = mgr.AABB_on_atoms(ase_atoms_repeat)
    # nrepeats = np.prod(mgr.get_repetitions_to_meet_cutoff(ase))

    # ase_atom_repeat = mgr.get_aseatoms_to_meet_cutoff(ase_atoms, cutoff)
    # forcefield_mixed_repeat = np.tile()
    # * get region indices for the regions in the group
    prop_labels = np.array([prop['label'] for prop in props])
    prop_indices = [np.where(prop_labels==g)[0][0] for g in group]
    
    # * this group has more than one entry, assuming it is a pbc group
    if len(group)>1:

        #* grid indices of all the points present in this group
        coords_indices = np.vstack([props[pi]['coords'] for pi in prop_indices])


        # * get the actual grid point coordinates from indices for the pbc_group
        coords =mgr.find_coord_from_indices(coords_indices, shape, ase_atoms) 

        # * get the grid point energies for the points in the group
        energies = mgr.gpe_aabb(coords, ab_repeat, forcefield_mixed_repeat, cutoff)

    else:

        pi = np.where(prop_labels==list( group )[0])[0][0]
        # print(pi)
        #* get coordinates and centroids
        coords_indices = props[pi]['coords']
        coords = mgr.find_coord_from_indices(coords_indices, shape, ase_atoms)
        # centroid = np.mean(coords, axis=0)
        # * histogram of the distances and bin centers
        energies = mgr.gpe_aabb(coords, ab_repeat, forcefield_mixed_repeat, cutoff)

    return energies

In [172]:
group = pbc_groups[1]
forcefield_mixed = mgr.mix_lorentz_berthelot(mat_atoms)
shape = regions.shape
props = regionprops(regions)

energies = get_energies_for_group(group, props,mat_atoms, forcefield_mixed, shape, cutoff=12.5)




In [20]:
import networkx as nx 
bins_energy = np.linspace(-3000, 100, 100)
types_list = [rag2.nodes[list(g)[0]]['cluster'] for g in pbc_groups]

# list(nx.get_node_attributes(rag2, 'cluster').values())
ctypes, cindices = np.unique(types_list,return_inverse=True)
# for c in ctypes:
types_indices = [np.where([t==c for t in types_list])[0] for c in ctypes]
types_counts = [len(t) for t in types_indices]
type_selection = [np.random.choice(ti , 3, replace=False) if types_counts[i]>3 else ti  for i,ti in enumerate(types_indices)]

import itertools  
merged_type_selection = [list(itertools.chain(*[list(pbc_groups[t]) for t in ts])) for ts in type_selection]
energies_list = [get_energies_for_group(mts, props, mat_atoms, forcefield_mixed, regions.shape) for mts in merged_type_selection]
histograms_bins= [np.histogram(e, bins=bins_energy, density=True) for e in energies_list]
ehistograms = [hb[0] for hb in histograms_bins]
ebincenters = [(hb[1][:-1]+hb[1][1:])/2 for hb in histograms_bins]
ehists_data = [ehistograms[cindices[i]] for i,g in enumerate(pbc_groups) for gg in list(g)]
ebins_data =  [ebincenters[cindices[i]] for i,g in enumerate(pbc_groups) for gg in list(g)] 
nx.set_node_attributes(rag2, dict(zip(rag2.nodes, ehists_data)), 'vdw_dist_y')
nx.set_node_attributes(rag2, dict(zip(rag2.nodes,ebins_data)), 'vdw_dist_x')


# [np.concatenate([get_energies_for_group(pbc_groups[s], props, mat_atoms, forcefield_mixed, regions.shape) for s in ts]) for ts in type_selection]
# for g in pbc_groups:
#     g = list(g)
#     print(np.vectorize(types_dict.get)(g))

In [22]:
rag2.nodes(data='vdw_dist_y')

NodeDataView({1: array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.000

In [246]:
def add_vdw_hist_to_rag(rag, props, pbc_groups, ase_atoms, forcefield_mixed, bins_energy=None , cutoff=12.5, minimum_npockets_per_type=3, add_energies=False):

    import numpy as np 
    import networkx as nx
    if 'cluster' not in rag.nodes[1].keys():
        raise ValueError('RAG must have a cluster attribute. Follow examples in docs to add clustering to RAG')
    else:
        types_list = [rag.nodes[list(g)[0]]['cluster'] for g in pbc_groups]

        if bins_energy is None:
            bins_energy = np.linspace(-3000, 100, 100)
        
        # list(nx.get_node_attributes(rag2, 'cluster').values())
        ctypes, cindices = np.unique(types_list,return_inverse=True)
        # for c in ctypes:
        types_indices = [np.where([t==c for t in types_list])[0] for c in ctypes]
        types_counts = [len(t) for t in types_indices]
        type_selection = [np.random.choice(ti , minimum_npockets_per_type, replace=False) if types_counts[i]>3 else ti  for i,ti in enumerate(types_indices)]

        import itertools  
        merged_type_selection = [list(itertools.chain(*[list(pbc_groups[t]) for t in ts])) for ts in type_selection]
        energies_list = [get_energies_for_group(mts, props, ase_atoms, forcefield_mixed, regions.shape) for mts in merged_type_selection]
        histograms_bins= [np.histogram(e, bins=bins_energy, normed=True) for e in energies_list]
        ehistograms = [hb[0] for hb in histograms_bins]
        ebincenters = [(hb[1][:-1]+hb[1][1:])/2 for hb in histograms_bins]
        ehists_data = [ehistograms[c] for c in cindices]
        ebins_data = [ebincenters[c] for c in cindices]
        nx.set_node_attributes(rag, dict(zip(rag.nodes, ehistograms)), 'vdw_hist_y')
        nx.set_node_attributes(rag, dict(zip(rag.nodes, ebincenters)), 'vdw_hist_x')
        if add_energies:
            energies_nodes = [energies_list[c] for c in cindices]
            nx.set_node_attributes(rag, dict(zip(rag.nodes, energies_nodes)), 'vdw_energies')
        
    return rag

In [247]:
# rag2.nodes[1].keys()
rag2= add_vdw_hist_to_rag(rag2, props, pbc_groups, mat_atoms, forcefield_mixed, bins_energy=None , cutoff=12.5, minimum_npockets_per_type=3)

C:\Users\ARUNGO~1\AppData\Local\Temp/ipykernel_22832/24831468.py:23: VisibleDeprecationWarning:

Passing `normed=True` on non-uniform bins has always been broken, and computes neither the probability density function nor the probability mass function. The result is only correct if the bins are uniform, when density=True will produce the same result anyway. The argument will be removed in a future version of numpy.



In [248]:
rag2.nodes[1]['vdw_hist_y']

array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 1.96204079e-05,
      

In [212]:
import itertools  
[list(itertools.chain(*[list(pbc_groups[t]) for t in ts])) for ts in type_selection]

[[2, 3, 4, 5], [1, 6]]

In [193]:
np.random.choice([2,4],3, replace=False)

ValueError: Cannot take a larger sample than population when 'replace=False'

In [187]:
rag2.nodes[1]

{'labels': [1],
 'pixel count': 4756,
 'total color': array([10114.75880188, 10114.75880188, 10114.75880188]),
 'mean color': array([2.1267365, 2.1267365, 2.1267365]),
 'centroid': array([ 0.98137708, 15.07243443,  3.8856246 ]),
 'centroid_indices': (4.203532380151388, 60.5, 16.0811606391926),
 'maxima': array([ 0.44918801, 15.197     ,  4.10763997]),
 'maxima_indices': array([ 2, 61, 17], dtype=int64),
 'maxima_radii': 3.5738101393121537,
 'psd_y': array([0.02266611, 0.15934959, 0.3640314 , 0.47942248, 0.35647603,
        0.16415756, 0.0796748 , 0.00755537, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
   

In [184]:
ctypes

array(['0', '1'], dtype='<U1')

In [167]:
ase_atoms = mat_atoms
cutoff = 12.5
group = pbc_groups[0]
forcefield_mixed = mgr.mix_lorentz_berthelot(mat_atoms)
shape = regions.shape
props = regionprops(regions)
ase_atoms_repeat, forcefield_mixed_repeat = mgr.repeat_to_meet_cutoff(ase_atoms, forcefield_mixed, cutoff)
# import numpy as np  

# * get aabb object on repeated system
ab_repeat = mgr.AABB_on_atoms(ase_atoms_repeat)
# nrepeats = np.prod(mgr.get_repetitions_to_meet_cutoff(ase))

# ase_atom_repeat = mgr.get_aseatoms_to_meet_cutoff(ase_atoms, cutoff)
# forcefield_mixed_repeat = np.tile()
# * get region indices for the regions in the group
prop_labels = np.array([prop['label'] for prop in props])
prop_indices = [np.where(prop_labels==g)[0][0] for g in group]

# # * this group has more than one entry, assuming it is a pbc group
if len(group)>1:

    #* fractional coordinates of all the points present in this group
    coords_indices = np.vstack([props[pi]['coords'] for pi in prop_indices])


    # * get the actual grid point coordinates from indices 
    coords = mgr.find_coord_from_indices(coords_indices, shape, ase_atoms)


    # * get the grid point energies for the points in the group
    import dask.array as da 
    energies = da.from_array(coords, chunks=(10000,3)).map_blocks(mgr.gpe_aabb, ab_repeat, forcefield_mixed_repeat, cutoff, dtype=np.float64,drop_axis=1).compute(scheduler='threads')
    # energies = mgr.gpe_aabb(coords, ab_repeat, forcefield_mixed_repeat, cutoff)

In [154]:
coords1 = mgr.find_coord_from_indices(props[0]['coords'], regions.shape, mat_atoms)
coords1.max(axis=0)

array([ 3.81364471, 19.43222951,  7.00715054])

In [166]:
energies.min()

-1383.3901258099756

In [12]:
props[0].indices

AttributeError: '<class 'skimage.measure._regionprops.RegionProperties'>' object has no attribute 'indices'

In [ ]:
def add_vdwhist_to_rag_with_pbc(rag, props, pbc_groups, bins, ase_atoms, shape, cutoff=12.5):



    # def get_pbc_distances(coords, centroid_pbc, ase_atoms):
    #     """
    #     > The function takes the coordinates of the grid points in the region, the centroid of the region, and
    #     the atoms object, and returns the distances of the grid points from the periodic region centroid
        
    #     :param coords: the coordinates of the atoms in the cluster
    #     :param centroid_pbc: the centroid of the cluster in the PBC frame
    #     :param ase_atoms: ASE atoms object
    #     :return: The distances between the centroid and the atoms in the cluster.
    #     """
        
    #     from ase.geometry import complete_cell 
    #     cell = complete_cell(ase_atoms.get_cell()).T
    #     # cell_inv = np.linalg.inv(cell) 
    #     df = coords - centroid_pbc
    #     df = (df - np.round(df))
    #     distances = np.linalg.norm(np.dot(cell, df.T).T, axis=1)
    #     return distances

    from scipy import stats 
    import numpy as np 
    import copy
    from .helper import find_coord_from_indices
    # shape = regions.shape
    G = copy.deepcopy(rag)
    prop_labels = np.array([prop['label'] for prop in props])
    
    #* get teh aabb object
    # * but to do this correctly, we need to repeat the unit cel enough times along each axis ot cover twice the cutoff distance
    
    from ase.build import make_supercell 
    P = np.eye(3)
    repeats =mgr.get_repetitions_to_meet_cutoff(ase_atoms, cutoff=cutoff)
    nrepeats = np.prod(repeats)
    np.fill_diagonal(P, val=list(repeats))


    # * let's repeat everything
    ase_atoms_repeat = make_supercell(ase_atoms, P)
    positions =ase_atoms_repeat.get_positions()
    cell = ase.geometry.complete_cell(ase_atoms_repeat.get_cell()).T
    forcefield_mixed_repeat  = np.tile(forcefield_mixed, (nrepeats,1))
    ab = mgr.AABB_on_atoms(ase_atoms)
    
    for group in pbc_groups: 
        prop_indices = [np.where(prop_labels==g)[0][0] for g in group]
        # print(group, prop_indices)
        if len(group)>1:
            #* fractional coordinates of all the points present in this group
            coords = np.vstack([props[pi]['coords'] for pi in prop_indices])/shape 
            # # print(coords.shape)
            # #* pbc centroid of the group in fractional coordinates
            # centroid = stats.circmean(coords, axis=0, low=0, high=1)
            # #* distances for the group in cartesian units
            # distances = get_pbc_distances(coords, centroid, ase_atoms)
            # # * histogram of the distances and bin centers

            # * get the actual grid point coordinates from indices 
            coords = np.vstack([find_coord_from_indices(props[pi]['indices'], shape, ase_atoms) for pi in prop_indices])



            # * get the grid point energies for the points in the group
            energies = mgr.gpe_aabb(coords, ab, forcefield_mixed_repeat, cutoff)

            vdw_y_pbc, bine = np.histogram(energies, bins=bins, density=True)
            vdw_x_pbc = (bine[1:] + bine[:-1])/2
            # * Now update the property of all the nodes in the graph with the pbc psd and cetnroid properties
            for i, g in enumerate(group):
                G.nodes[props[prop_indices[i]]['label']]['centroid_pbc'] = find_coord_from_indices(indices = centroid*shape, ase_atoms=ase_atoms, shape=shape)
                G.nodes[props[prop_indices[i]]['label']]['vdw_x_pbc'] = vdw_x_pbc
                G.nodes[props[prop_indices[i]]['label']]['vdw_y_pbc'] = vdw_y_pbc
        else:
            pi = np.where(prop_labels==list( group )[0])[0][0]
            # print(pi)
            #* get coordinates and centroids
            coords = props[pi]['coords']/shape
            coords = mgr.find_coord_from_indices(props[pi]['indices'], shape, ase_atoms)
            # centroid = np.mean(coords, axis=0)
            # * histogram of the distances and bin centers
            energies = mgr.gpe_aabb(coords, ab, forcefield_mixed_repeat, cutoff)
            psd_y_pbc, bine = np.histogram(distances, bins=bins, density=True)
            psd_x_pbc = (bine[1:] + bine[:-1])/2

            # * Now update the property of all the nodes in the graph with the pbc psd and cetnroid properties
            G.nodes[props[pi]['label']]['centroid_pbc'] = find_coord_from_indices(indices = centroid*shape, ase_atoms=ase_atoms, shape=shape)
            G.nodes[props[pi]['label']]['psd_x_pbc'] = psd_x_pbc
            G.nodes[props[pi]['label']]['psd_y_pbc'] = psd_y_pbc
    
    return G